In [11]:
import pandas as pd
import numpy as np
ml_table = pd.read_csv('ML_TABLE_LABELED.csv')
print(ml_table.shape)
ml_table.head(2)

(96476, 83)


,order_id,customer_id,order_status,order_purchase,order_approved,order_delivered_carrier,order_delivered_customer,order_estimated_delivery,num_sellers,num_items,...,pay_debit_card,pay_not_defined,pay_voucher,customer_unique_id,customer_zipcode,customer_city,customer_state,customer_lat,customer_lng,on_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1.0,1.0,...,0.0,0.0,2.0,7c396fd4830fd04220f754e42b4e5bff,3149.0,sao paulo,SP,-23.576983,-46.587161,1
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1.0,1.0,...,0.0,0.0,0.0,af07308b275d755c9edb36a90c618231,47813.0,barreiras,BA,-12.177924,-44.660711,1


In [12]:
ml_table=ml_table.drop(columns=['order_id','customer_id','customer_unique_id','customer_zipcode','seller_zipcode','customer_city','seller_city'
,'max_shipping_limit_date', 'order_delivered_carrier', 'order_delivered_customer', 'order_approved',
'order_status']) # future data must be dropped

# one-hot encodeing for customer_state
ml_table = pd.get_dummies(ml_table, columns=['customer_state'], prefix='cust_state',dtype=int) 

In [13]:
ml_table.head(2)

,order_purchase,order_estimated_delivery,num_sellers,num_items,total_price,total_freight_value,avg_seller_lat,avg_seller_lng,max_product_weight_grams,max_product_length_cm,...,cust_state_PR,cust_state_RJ,cust_state_RN,cust_state_RO,cust_state_RR,cust_state_RS,cust_state_SC,cust_state_SE,cust_state_SP,cust_state_TO
0,2017-10-02 10:56:33,2017-10-18,1.0,1.0,29.99,8.72,-23.680729,-46.444238,500.0,19.0,...,0,0,0,0,0,0,0,0,1,0
1,2018-07-24 20:41:37,2018-08-13,1.0,1.0,118.70,22.76,-19.807681,-43.980427,400.0,19.0,...,0,0,0,0,0,0,0,0,0,0


In [14]:
ml_table['order_purchase']=pd.to_datetime(ml_table['order_purchase'])
ml_table['order_estimated_delivery']=pd.to_datetime(ml_table['order_estimated_delivery'])

print(ml_table.describe())

                   order_purchase    order_estimated_delivery   num_sellers  \
count                       96476                       96476  96476.000000   
mean   2018-01-01 22:44:47.156474  2018-01-25 16:27:14.993158      1.013900   
min           2016-09-15 12:16:38         2016-10-04 00:00:00      1.000000   
25%           2017-09-14 08:10:58         2017-10-05 00:00:00      1.000000   
50%           2018-01-20 19:21:46         2018-02-16 00:00:00      1.000000   
75%    2018-05-05 18:28:21.750000         2018-05-28 00:00:00      1.000000   
max           2018-08-29 15:00:37         2018-10-25 00:00:00      5.000000   
std                           NaN                         NaN      0.123538   

          num_items   total_price  total_freight_value  avg_seller_lat  \
count  96476.000000  96476.000000         96476.000000    96263.000000   
mean       1.142212    137.038176            22.785442      -22.794311   
min        1.000000      0.850000             0.000000      -32.07

In [15]:
####################   trade-off of time-based splitting vs random+stratified
## here ratio between on-time and late is not 92/8 across sets but time-based splitting it also measure
## that if there is an improvement in the deleivery in the future.
## so when ai is tested by future test data, so the accuracy will be real.

ml_table_sorted = ml_table.sort_values('order_purchase').reset_index(drop=True)

n = len(ml_table_sorted)
train_end = int(n * 0.70)
val_end = int(n * 0.80)

train = ml_table_sorted.iloc[:train_end]
val = ml_table_sorted.iloc[train_end:val_end]
test = ml_table_sorted.iloc[val_end:]
print(f"n is {n}")
print(train['order_purchase'].min(), train['order_purchase'].max())
print(val['order_purchase'].min(), val['order_purchase'].max())
print(test['order_purchase'].min(), test['order_purchase'].max())
print(train['on_time'].value_counts(normalize=True))
print(val['on_time'].value_counts(normalize=True))
print(test['on_time'].value_counts(normalize=True))

n is 96476
2016-09-15 12:16:38 2018-04-15 20:07:56
2018-04-15 20:10:23 2018-05-26 16:54:05
2018-05-26 17:57:44 2018-08-29 15:00:37
on_time
1    0.909718
0    0.090282
Name: proportion, dtype: float64
on_time
1    0.926506
0    0.073494
Name: proportion, dtype: float64
on_time
1    0.947087
0    0.052913
Name: proportion, dtype: float64


In [16]:
train.to_csv('ml_train.csv',index=False)
val.to_csv('ml_val.csv',index=False)
test.to_csv('ml_test.csv',index=False)